In [1]:
import joblib

bow_mb = joblib.load('../models/bow_mb_ceas.pkl')
X_test_bow_mb = joblib.load('../models/X_test_bow_mb_ceas.pkl')
y_test_bow_mb = joblib.load('../models/y_test_bow_mb_ceas.pkl')

tf_idf_svm = joblib.load('../models/tf_idf_svm_ceas.pkl')
X_test_tf_idf_svm = joblib.load('../models/X_test_tf_idf_svm_ceas.pkl')
y_test_tf_idf_svm = joblib.load('../models/y_test_tf_idf_svm_ceas.pkl')

w2v_rf = joblib.load('../models/w2v_rf_ceas.pkl')
X_test_w2v_rf = joblib.load('../models/X_test_w2v_rf_ceas.pkl')
y_test_w2v_rf = joblib.load('../models/y_test_w2v_rf_ceas.pkl')

sbert_lr = joblib.load('../models/sbert_lr_ceas.pkl')
X_test_sbert_lr = joblib.load('../models/X_test_sbert_lr_ceas.pkl')
y_test_sbert_lr = joblib.load('../models/y_test_sbert_lr_ceas.pkl')


In [2]:
pred_bow_mb = bow_mb.predict(X_test_bow_mb)
pred_tf_idf_svm = tf_idf_svm.predict(X_test_tf_idf_svm)
pred_w2v_rf = w2v_rf.predict(X_test_w2v_rf)
pred_sbert_lr = sbert_lr.predict(X_test_sbert_lr)

In [3]:
model_predictions = [
('BOW + Muiltinomial Bayes', pred_bow_mb,y_test_bow_mb),
('TF/IDF + SVM', pred_tf_idf_svm,y_test_tf_idf_svm),
('Word2Vec + RandomForrest',pred_w2v_rf,y_test_w2v_rf),
('SBERT + Logistic Regression',pred_sbert_lr,y_test_sbert_lr),
]

In [4]:
from sklearn import metrics

def get_model_metrics(y_true, y_pred, model):

    report = metrics.classification_report(y_true, y_pred, output_dict=True)

    b_acc = metrics.balanced_accuracy_score(y_true, y_pred)

    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']

    results = {
        'Model': model,
        'Macro Precision': macro_avg['precision'],
        'Macro Recall': macro_avg['recall'],
        'Macro F1-Score': macro_avg['f1-score'],
        'Weighted Precision': weighted_avg['precision'],
        'Weighted Recall': weighted_avg['recall'],
        'Weighted F1-Score': weighted_avg['f1-score'],
        'Balanced Accuracy': b_acc
    }

    return results

In [5]:
import pandas as pd
all_results = []
for model_name, model_pred, y_test in model_predictions:
    model_res = get_model_metrics(y_test, model_pred, model_name)
    all_results.append(model_res)

all_results_df = pd.DataFrame(all_results)
all_results_df.set_index("Model", inplace=True)
all_results_df = all_results_df.apply(lambda x: round(x, 4))

print("Model Performance Comparison:")
print(all_results_df)

Model Performance Comparison:
                             Macro Precision  Macro Recall  Macro F1-Score  \
Model                                                                        
BOW + Muiltinomial Bayes              0.8716        0.8604          0.8646   
TF/IDF + SVM                          0.9725        0.9681          0.9701   
Word2Vec + RandomForrest              0.9674        0.9647          0.9660   
SBERT + Logistic Regression           0.9824        0.9814          0.9819   

                             Weighted Precision  Weighted Recall  \
Model                                                              
BOW + Muiltinomial Bayes                 0.8702           0.8695   
TF/IDF + SVM                             0.9712           0.9709   
Word2Vec + RandomForrest                 0.9669           0.9668   
SBERT + Logistic Regression              0.9823           0.9823   

                             Weighted F1-Score  Balanced Accuracy  
Model                   

Najveći fokus u ovoj analizi ću posvetiti odzivu i preciznosti modela. Preciznost mi pokazuje koliko je mejlova tačno klasifikovano u odnosu na sve klasifikovane, dok odziv procentualno poredi koliko je tačno klasifikovano mejlova u odnosu na ukupne mejlove

Želim samo da naglasim da svi ovi modeli imaju jako dobre rezultate, ali mislim da je to rezultat dataset-a, koji je pun duplikata, specijalnih zastarelih mime formata i gibberish-a, tako da je moguće da su modeli našli neku prečicu u klasifikaciji. Ja sam se potrudio da izbegnem ovo izbacivanjem duplikata i čišćenjem tekstualnih podataka nakon kreiranja obeležja, ali opet možda postoji nešto što sam propustio

S obzirom da je Bow imao ograničen rečnik on je i imao najmanju preciznost, ovo ne znači nužno da je ova metoda najgora, mislim da bi ona davala bolje rezultate kad bi se modeli testirali na savremenim modelima, zato što njegov rečnik nije vezan za termine ovog caes dataset-a

Iako SBERT ima najveći odziv i pouzdanost, bojim se da u rečenicama nije ostao neki skriveni zajendički kontekst(neki specijalni karakteri) na osnovu koga je model naučio da klasifikuje mejlove. Na primer ukoliko su postojali mejlovi koji se obraćaju specijalnoj osobi, ovaj model bi prepoznao ovo kao spam. U svakom slučaju mislim da ova kombinacija predstavlja jako dobar model nad čistijim podacima

TF/IDF + SVM i W2V + RF daju jako slične rezultate. U listi najbitnijih prediktora za TF/IDF su bile sve logične reči koje se mogu smatrati kao indikatori spama, dok sa druge strane Word2Vec omogućava kontekstno uporedjivanje reči

Poređenjem Makro i Weighted vrednosti odziva i preciznosti zaključujem da mali disbalans u broju uzoraka klasa ne remeti rezultate modela. Model pokazuje konzistente rezultate iako nije potpuno ravnomerna podela uzorka

Sobzirom da je fokus na preciznosti i odzivu daću blagu prednost kombinaciji TF/IDF + SVM